In [0]:
from pyspark.sql.functions import *

In [0]:
def read_source_and_load_into_bronze_as_delta_table(schema, schema_path, source_path, checkpoint_path, target_table):
    df = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .schema(schema)
        .option("cloudFiles.schemaLocation", schema_path)
        .load(source_path))
        
    df_bronze = (df
        .withColumn("_ingestion_ts", current_timestamp())
    )

    (df_bronze.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(target_table)
    )